# mesh02 — 도구 상자: 어떤 라이브러리를 왜 골랐고, 검사기는 무엇을 보나

> ⚠ **이 노트북은 생성물이다.** 수정은 `report_mesh/src/make_mesh02.py` 에서 하고
> 재실행할 것(`.ipynb` 를 직접 고치면 다음 빌드에서 사라진다).

**이 편이 답하는 질문** — 이 드론들을 만드는 도구는 무엇이고, 그 도구로 만든 것을 무엇이 검사하며, 그 검사가 아직 못 보는 것은 무엇인가.

**무엇을 근거로 하는가** (본문 수치는 손으로 적지 않고 아래 **원장** — 검사기가 낸 측정 기록 파일 — 에서 주입한다)

| 원장 | 무엇이 들어 있나 |
|---|---|
| `outputs/mesh_inspect_materials_check_0816.json` | 재질 배정·검사기·프롭 외 부품 감사(13그룹·10종) |
| `outputs/mesh_inspect_body_arms_0816.json` | 동체·팔·다리·모터 벨 전수 실측(10종) + matrice4e 공식 CAD 정정 착지 검증 |
| `docs/MESH_AUDIT_0816.md` | 적대적 감사 — 발견·반증·수리 우선순위(§⑤) |
| `report_mesh/outputs/mesh_verify_canon_0817.json` | ⭐**정본 판** 기하 검증 A·B·C·D·F·G |
| `report_mesh/outputs/mesh_canon_0817.json` | ⭐**정본 판** 스위치·예산 스냅샷·재질 가중 매몰면 |
| `docs/MESH_CERTIFICATE.md` | 메쉬 인증서 — 검사 체계가 무엇을 장담하고 무엇은 못 하는가 |
| `report_mesh/outputs/mesh_verify.json` | 기하 검증 스위트 A~I — H·I(GPU 절)는 아직 이쪽만 있다 |

**한 줄 요약** — 드론 10기의 CAD 메쉬(총 삼각형 301,506개)는 딱 5개 라이브러리
(numpy · shapely · trimesh · manifold3d · scipy) 위에서 만들어지고, 결과는 경량 컨테이너
(`src/geom.py` 의 `Mesh`)에 담겨 뒷단 파이프라인으로 흘러간다.
이 편에서는 **각 도구가 무엇을 하고, 왜 그것을 골랐고, 어떤 방식으로 쓰는지**를 하나씩 뜯어본다.

⭐ **이 편이 설명하는 메쉬는 «정본 판»이다** — 파일명 꼬리표 `_mfixbatteryi5_blperairframe`. 어떤 기본값이
그 판을 정하는지는 §0.2 에 있다.

| 용어 | 한 줄 풀이 |
|---|---|
| 메쉬(mesh) | 3D 모양을 **삼각형 조각들의 모음**으로 표현한 것 (꼭짓점 목록 + 삼각형 목록) |
| 정점(vertex) / 면(face) | 3D 점 (x,y,z) 하나 / 정점 3개를 이은 삼각형 하나 |
| watertight(방수) | 메쉬에 구멍·틈이 하나도 없어 물을 부어도 안 새는 상태 — 부피 계산·불리언의 전제 |
| 불리언(CSG) | 두 입체의 합집합/차집합/교집합 — 레고 붙이기·조각칼로 파내기에 해당 |
| 로프트(loft) | 여러 **단면**을 배 늑골처럼 세워 놓고 겉껍질을 씌워 곡면을 만드는 기법 |
| 스윕(sweep) | 단면 하나를 **곡선 경로를 따라 밀어** 관·팔 모양을 만드는 기법 |
| 회전체(revolve) | 옆모습 곡선을 축 둘레로 한 바퀴 돌려 만든 입체 (도자기 물레와 같음) |
| 초타원(superellipse) | 타원과 직사각형의 중간 곡선 — 실제 드론 동체 단면이 이 모양 |
| NACA 익형 | 미 항공자문위(NACA)의 표준 날개 단면 공식 — 프로펠러 단면에 사용 |
| 법선(normal) | 면이 바라보는 수직 방향 화살표 — 안/밖 구분과 전자기 계산의 기준 |
| 퇴화 삼각형(degenerate) | 세 점이 겹치거나 일직선이라 **넓이가 0**인 불량 삼각형 |
| PO / SBR | 물리광학 / 슈팅&바운싱 레이 — 이 메쉬를 소비하는 레이더 반사(RCS) 계산법 |
| 정본(canon) 판 | 지금 기본으로 지어지는 형상. 스위치를 아무것도 안 주면 이 판이 나온다(§0.2) |
| 파일명 꼬리표 | 산출물 이름 끝에 붙는 표시. 어느 판에서 계산한 결과인지 이름만으로 갈린다 |
| 예산(budget) | 검사가 «이만큼까지는 통과» 로 선언해 둔 값. «옳다» 가 아니라 «지금 이만큼이다» (§8.2) |
| 매몰면(buried face) | 다른 부품 **속**에 들어가 실물이라면 안 보이는 삼각형. PO 는 가림을 안 보므로 두 번 센다 |
| 양성 대조 | 일부러 결함을 심어 «검사가 정말 무는가» 를 보는 시험. 음성 대조(멀쩡한 표본에 0)의 짝이다 |
| 골든 봉인 | 오늘 형상의 지문을 떠 두고 내일과 대조하는 장치. 지키는 것은 «안 바뀜» 이지 «옳음» 이 아니다 |

## 0. 큰 그림 — 도구는 두 층이다

이 프로젝트의 3D 도구는 역할이 뚜렷한 **두 층**으로 되어 있다.

1. **`src/geom.py` — Mesh 컨테이너 + 챔버·범용 프리미티브.** numpy 만 쓰는 경량 모듈로,
   메쉬를 담는 그릇(`.v`/`.f`/`.g`)과 챔버·데모용 기본 도형(box/cylinder/uv_sphere/
   pyramid_field 등)을 제공한다. **드론 제작 도구가 아니다.**
2. **`src/cadkit.py` — 라이브러리를 제대로 쓰는 CAD 툴킷.** trimesh + manifold3d + shapely +
   scipy 로 로프트·스윕·회전체·불리언·검증을 수행한다. 실제 드론 형상(눈물방울 동체, 익형
   프로펠러)은 이 층 위의 `src/drone_cad.py` 가 만든다.

메쉬 엔진은 `"trimesh+manifold3d (drone_cad)"` 단일 경로다 ← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` `_meta.mesh_engine`,
src/drones.py (`_build_frame_raw` — "CAD(trimesh+manifold3d 로프트/불리언) 단일 경로").
즉 **모양을 계산하는 본체는 cadkit(라이브러리) 층**이고, geom.Mesh 는 결과를 담아
뒷단 파이프라인(장면 구성·RCS·마이크로도플러)으로 넘기는 **공용 컨테이너**다
← 출처: src/cadkit.py ("마지막에 geom.Mesh 로 변환한다 → 기존 파이프라인이 그대로 돈다").

왜 층을 나누나? — 드론 형상에는 불리언·로프트·스무딩·검증이라는 무거운 CAD 연산이 필요하고
(§2.5), 챔버 벽·흡수체 피라미드 같은 단순 형상과 파이프라인 인터페이스에는 그게 필요 없기
때문이다. 무거운 쪽은 라이브러리에 맡기고, 가벼운 쪽은 투명한 자작 코드로 유지한다.

### 0.1 이 노트북이 쓰는 실제 버전 — 그리고 왜 버전이 이 편의 주제인가

아래 셀은 지금 커널(py312)에 실제로 설치된 버전을 출력한다. 리포트 생성 시점에 확인된 버전은
numpy 2.5.2 · **trimesh 5.0.0** · manifold3d 3.5.2 ·
shapely 2.1.2 · scipy 1.18.0 였다
← 출처: 실제 설치 환경(`/workspace/.venvs/py312`, importlib.metadata 로 조회).

⭐ **버전을 굵게 적는 이유** — 라이브러리 판이 바뀌면 **검사의 뜻이 바뀔 수 있다.**
trimesh 5.0.0 에서 `split()` 의 `repair` 기본값이 **켜짐**이다. 그냥 부르면
구멍 뚫린 부품을 **조용히 메운 사본**을 돌려주고, 그 사본은 당연히 «수밀» 로 나온다.

확인은 세 줄이면 된다 — 삼각형 1장을 뺀 상자를 두 방식으로 쪼개 보는 것이다(§8 셀).
그래서 이 저장소의 검사 경로는 **모든 `split` 에 `repair=False` 를 명시**한다
← 출처: `src/mesh_check.py` 머리말·`_split()`, `report_mesh/src/verify_mesh_suite.py` 머리말.

이것이 이 편의 교훈이다: **도구 편에서 버전은 각주가 아니라 본문이다.**

In [ ]:
# trimesh 5.x 의 split 기본값 확인 — 검사기가 «수리한 사본» 을 보지 않게 하는 이유
import trimesh
b = trimesh.creation.box()
holed = trimesh.Trimesh(vertices=b.vertices, faces=b.faces[:-1], process=True)  # 삼각형 1장 뺌
for kw in [{}, {"repair": False}]:
    c = holed.split(only_watertight=False, **kw)[0]
    tag = "기본값" if not kw else "repair=False"
    print(f"{tag:12s} 면 {len(c.faces):3d}  수밀 {str(c.is_watertight):5s}  부피 {abs(c.volume):.3f}")
print("→ 기본값은 없는 삼각형을 지어 구멍을 메운다. 검사기가 하면 안 되는 일이다.")

In [ ]:
# 설치된 라이브러리 버전 확인 — 재현 시 아래 값이 본문 값과 같은지 보라
import importlib.metadata as im
for pkg in ["numpy", "trimesh", "manifold3d", "shapely", "scipy"]:
    print(f"{pkg:12s} {im.version(pkg)}")

### 0.2 ⭐ 어느 «판» 을 짓는지는 두 기본값이 정한다

이 편의 결론은 «도구는 능력을 주지만 **기본값이 규약을 정하지는 않는다**» 다(§10).
그러니 지금 기본값이 무엇인지부터 적는다. 형상의 판(version)을 고르는 스위치가 둘 있고,
둘 다 **의존성이 없는 맨 아래층**인 `src/geom.py` 한 곳에 있다 — 스위치가 두 군데 있으면
«켰는데 반만 켜지는» 사고가 나기 때문이다 ← 출처: `src/geom.py` 스위치 블록 머리말.

| 스위치 | 지금 기본값(정본) | 무엇이 달라지나 | 옛 판으로 되돌리는 법 |
|---|---|---|---|
| `geom.MESH_FIX_CANON` | `battery, i5` | `battery` = 배터리 팩 상자와 구조판 상자가 서로 파고든 것을 불리언 합집합으로 없앤다(4기체) · `i5` = mini2 셸의 구멍을 닫는다 | `MESH_FIX=none` |
| `geom.BLADE_LAW_CANON` | `per_airframe` | 기체마다 **그 기체의 순정 프로펠러** 평면형을 쓴다 | `BLADE_LAW=legacy` |
| 파일명 꼬리표 | `_mfixbatteryi5_blperairframe` | 정본 판 산출물의 이름에 붙는다 | 옛 판은 꼬리표가 **없다** — 그래서 두 판이 이름만으로 갈린다 |

| 무엇 | 어디 |
|---|---|
| `MESH_FIX_CANON` (정본 수리 목록) | src/geom.py:92 |
| `BLADE_LAW_CANON` (정본 날 법칙) | src/geom.py:105 |
| `mesh_fix_set()` / `mesh_fix_enabled()` | src/geom.py:113, 129 |
| `blade_law_canon()` | src/geom.py:108 |
| `set_mesh_fix()` (코드에서 켜기) | src/geom.py:135 |

(행 번호는 이 노트북을 만들 때 소스에서 직접 읽는다 — 손으로 적지 않는다.)

⭐ **판정은 호출 시점에 한다.** `geom.mesh_fix_set()`·`geom.blade_law_canon()` 이 환경변수를 **부를 때마다** 읽으므로, import 뒤에 켜도 듣는다 ← 출처: `src/geom.py` 두 함수의 docstring.

⭐ **꼬리표가 규약인 이유** — 정본 판 산출물은 이름에 `_mfixbatteryi5_blperairframe` 가 붙는다. 이름이 같으면 계산기가 **옛 판 결과를 재사용**하고, 재계산이 «건너뜀» 으로 끝나 버린다 ← 출처: `benchmark/elevation_sweep_md.py` 꼬리표 블록.

⛔ `MESH_FIX=none BLADE_LAW=legacy` 를 주면 옛 판이 **비트동일**하게 다시 나온다 ← 출처: `benchmark/regress_blade_law_bitidentical.py` · `src/mesh_check.py` legacy 회귀. 전환 직전 산출물은 `/data/public/sionna/archive_pre_meshfix_20260817/` 에 있다(꼬리표 없는 샤드 3,813개 + README).

## 1. numpy — 모든 좌표 계산의 기반

**무엇** — 수치 배열 라이브러리(버전 2.5.2 ← 출처: 실제 설치 환경). 이 프로젝트에서
정점 좌표는 전부 `(N, 3)` 모양의 numpy 배열이고, 이동·회전·확대는 4×4 행렬 곱 한 번이다
← 출처: src/geom.py (`Mesh.transformed` 가 동차좌표 `(N,4)` 를 만들어 `M @ P.T` 로 변환).

**왜** — 파이썬 반복문으로 정점 수만 개를 하나씩 옮기면 수백 배 느리다. numpy 는 C 로 구현된
벡터 연산이라 좌표 수만 개를 한 번에 처리한다. 사실상 대안이 없는 과학계산 표준이며(BSD
라이선스, numpy.org), 아래의 trimesh·shapely·scipy 도 모두 numpy 배열을 주고받는다 —
**공용 언어**인 셈이다.

**어떻게** — 이 프로젝트의 좌표 규약은 numpy 배열에 담긴 **z 축 위(up), 단위 미터(m)** 다.
드론 제원은 mm 로 들어오므로 /1000 해서 쓴다 ← 출처: src/geom.py (좌표계·단위 주석).

## 2. `src/geom.py` — Mesh 컨테이너와 챔버·범용 프리미티브

**무엇** — 이 프로젝트의 **공용 메쉬 컨테이너**. 메쉬를 딱 세 목록으로
표현한다: `.v`(정점 좌표), `.f`(삼각형 인덱스), `.g`(면별 **그룹 이름** = body/arm/motor/prop...)
← 출처: src/geom.py. 그룹 이름표는 나중에 부위별 색칠과 부위별 전파재질(Sionna
RadioMaterial) 부여에 쓰인다 ← 출처: src/geom.py. 컨테이너 외에 챔버·데모용
**범용 프리미티브**(직육면체·원기둥·구·흡수체 피라미드)도 제공한다 — 드론 형상 제작은
cadkit/drone_cad 층의 몫이고, geom.py 는 그 일을 하지 않는다(§0).

**왜 numpy 만으로 자작인가** — 소스 주석에 이유가 적혀 있다 ← 출처: src/geom.py:

> "**'도형이 어떻게 만들어지는지' 코드로 눈에 보이게** 하기 위해서입니다.
> (사용자가 쉽게 이해하는 것이 이 프로젝트의 1순위 목표)"

컨테이너와 단순 도형에는 무거운 CAD 연산이 필요 없으므로, 이 층은 **단순함·투명함·교육**을
우선한다. box 하나가 코드 20줄이라, 삼각형이 어떤 순서로 감기는지(winding) 눈으로 따라갈 수 있다.

**어떻게 — 제공하는 것들** ← 출처: src/geom.py 의 각 함수:

| 함수 | 만드는 것 | 소스 위치 |
|---|---|---|
| `Mesh` (클래스) | 컨테이너 — .v/.f/.g + 변환·바운즈·그룹 조회 | src/geom.py:153 |
| `box` | 직육면체(정점 8개, 삼각형 12개) | src/geom.py:296 |
| `cylinder` | 원기둥·원뿔대(`r_top`) | src/geom.py:325 |
| `pyramid` / `pyramid_field` | 전파흡수체 피라미드 1개 / 피라미드 밭 | src/geom.py:359, 442 |
| `uv_sphere` | 구 — 극점을 삼각형 팬으로 접어 **넓이 0 삼각형을 안 만든다** | src/geom.py:383 |
| `translate` / `rotate` / `scale` | 4×4 변환 행렬 | src/geom.py |
| `write_obj_per_group` | 그룹별 .obj 저장 — "OBJ 1개 = Sionna 재질 1개" 규약 | src/geom.py:249 |
| `mesh_fix_set` / `blade_law_canon` | ⭐**어느 판을 지을지**를 돌려주는 두 함수(§0.2) | src/geom.py:113, 108 |

(행 번호는 이 노트북을 만들 때 `inspect` 로 소스에서 직접 읽는다 — 손으로 적지 않는다.)

⭐ **왜 판 스위치가 하필 이 파일에 있나** — geom.py 는 의존성이 없는 맨 아래층이라 cadkit 도
drone_cad 도 검사기도 전부 import 할 수 있다. 진리원이 하나여야 «반만 켜지는» 사고가 안 난다
← 출처: `src/geom.py` 스위치 블록 머리말.

⚠ **`uv_sphere` 의 극점에 대해 정확히 적는다.** 넓이 0 삼각형은 **0개**가 맞다.
다만 극점 자리에 정점이 세그먼트 수만큼 **겹쳐** 있어, 출하 인덱스 그대로는 그 구가
수밀이 아니다(합쳐 보면 수밀이다). 선택 인자 `uv_sphere(..., weld_poles=True)` 를 주면
삼각형 좌표·개수·부피가 **완전히 같은 채로** 정점만 2·(seg−1)개 줄어든다. 기본값은 꺼져
있어 예전과 비트동일하다 ← 출처: `outputs/mesh_inspect_materials_check_0816.json`
`uv_sphere`·`selftest_weld_poles`.

In [ ]:
# geom.py 맛보기 — 챔버·범용 프리미티브를 라이브러리 없이 삼각형만으로
import os, sys, numpy as np
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
import geom

shapes = {"box":       geom.box(0.30, 0.20, 0.10),          # 직육면체 30x20x10 cm
          "cylinder":  geom.cylinder(0.05, 0.20, seg=24),   # 반지름 5 cm 원기둥
          "uv_sphere": geom.uv_sphere(0.05)}                # 반지름 5 cm 구
for name, m in shapes.items():
    lo, hi = m.bounds()
    size_cm = np.round((hi - lo) * 100, 1)
    print(f"{name:10s} 삼각형 {m.n_tris():4d}개  크기(cm) {size_cm}")

### 2.5 왜 드론은 geom.py 로 만들지 않나 — CAD 요구사항 4가지

실물 드론 외형에는 삼각형을 손으로 쌓는 방식이 감당 못 하는 연산 4가지가 필요하다.
`cadkit.py` 상단 주석이 그 목록이다 ← 출처: src/cadkit.py (요지):

* **불리언(CSG)** — 오목부·홈·리세스를 파고, 겹친 프리미티브의 **속에 파묻힌 면**을 없앤다.
* **로프트(loft)** — 단면이 변하는 매끈한 동체(눈물방울·허리 잘록)를 만든다.
* **스무딩/서브디비전** — 각진 다각기둥을 실물 같은 곡면으로 다듬는다.
* **검증** — watertight·법선 방향·winding 을 **기계가** 파트마다 확인한다. 눈으로는
  안쪽을 향한 법선 같은 결함을 놓치기 쉽다(렌더링에는 멀쩡해 보이는 경우가 많다).

왜 RCS(레이더 반사 면적)에 중요한가: 형상이 각지거나 내부 면이 남거나 법선이 뒤집히면
전자기 계산(PO/SBR)이 실물과 다른 면을 "보게" 된다 ← 출처: src/drone_cad.py ("RCS 는
외형(투영면적)과 재질 분포가 결정한다. 실루엣이 틀리면 σ 가 틀린다").

## 3. shapely — 2D 단면 폴리곤 공장

**무엇** — 2D 기하 라이브러리(버전 2.1.2 ← 출처: 실제 설치 환경). 폴리곤(다각형)의
생성·버퍼(테두리 넓히기)·보간을 담당한다. 내부적으로 검증된 C++ 기하 엔진 GEOS 를 쓴다
(BSD 라이선스, shapely.readthedocs.io).

**왜** — 3D 로프트/스윕의 입력은 결국 **2D 단면**이다. 단면을 다루는 연산 중 특히
"정확한 오프셋(둥근 모서리)"과 "외곽선 등간격 재샘플"은 직접 짜면 모서리·자기교차 처리에서
틀리기 쉽다. shapely 는 이를 `buffer` 와 `interpolate` 한 줄로 제공한다
← 출처: src/cadkit.py (`rounded_rect` — "shapely 의 buffer 로 만든다(정확한 오프셋)"),
src/cadkit.py (`_resample` — "로프트하려면 단면들의 점 수가 같아야 한다").

**어떻게** — 이 프로젝트의 핵심 단면 두 가지가 shapely `Polygon` 으로 만들어진다:

* `superellipse(a, b, n)` — **초타원**: n=2 면 타원, n>2 면 모서리가 둥근 직사각형.
  "실제 드론 단면은 순수 타원도 박스도 아니고 이 중간이다" ← 출처: src/cadkit.py
  docstring. 접이식 동체는 n≈2.9 를 쓴다 ← 출처: src/drone_cad.py (`_body_folding` 의
  `n_pow=2.9`).
* `rounded_rect(w, h, r)` — 둥근 모서리 직사각형: 접이식 팔(arm)의 단면 ← 출처:
  src/drone_cad.py (`_arm_folding` 이 이 단면을 스윕 경로에 태운다).

프로펠러 단면(NACA 익형)도 shapely Polygon 으로 조립된 뒤, 회전(피치각)·평행이동(스윕)을
`shapely.affinity` 로 처리한다 ← 출처: src/drone_cad.py (`_airfoil`, `_blade`).

## 4. trimesh — 메쉬의 표준 컨테이너이자 품질 검사관

**무엇** — 파이썬 3D 삼각형 메쉬 라이브러리(버전 5.0.0 ← 출처: 실제 설치 환경;
MIT 라이선스, github.com/mikedh/trimesh). 이 프로젝트에서 맡는 핵심 역할은 네 가지다
← 출처: src/cadkit.py ("메쉬 컨테이너 · 프리미티브 · 스무딩 · 서브디비전 · 검증"; 여기서
스무딩·서브디비전은 아래 "어떻게" 문단에서 함께 다룬다):

1. **컨테이너/IO** — 정점·면을 담고 OBJ 등으로 읽고 쓴다.
2. **프리미티브** — `creation.box/cylinder/icosphere/capsule` 을 cadkit 이 얇게 감싼다
   ← 출처: src/cadkit.py:457~479 (`box`, `cyl`, `sphere`, `capsule`).
3. **검증 API** — `is_watertight`(구멍 없음), `is_winding_consistent`(감김 일관), `volume` 부호
   (법선이 안쪽이면 부피가 음수), `nondegenerate_faces`(넓이 0 삼각형 탐지). 눈으로는
   놓치기 쉬운 기하 결함을 수치로 드러낸다 ← 출처: src/cadkit.py:252
   (`Assembly.check`).
   
   ⭐ **다만 API 를 주는 것과 올바르게 부르는 것은 다르다.** §0.1 에서 봤듯 `split()` 의
   `repair` 기본값이 켜져 있어서, 그냥 부르면 «검사하려던 그 결함» 이 사라진 사본을 본다.
   그래서 우리 검사 경로는 `repair=False` 를 명시한다.
4. **불리언 인터페이스** — `trimesh.boolean.union(..., engine="manifold")` 처럼 계산 엔진을
   갈아끼울 수 있는 표준 창구 ← 출처: src/cadkit.py `Assembly.union_group`.

**왜 trimesh 가 표준인가** — numpy 배열을 그대로 쓰는 가벼운 API, 순수 파이썬 + 선택적
가속, 그리고 위 검증 속성들이 property 하나로 제공된다. 검토했던 대안: **open3d**(포인트클라우드
·시각화 중심으로 무겁고 CSG 없음), **pymeshlab**(필터 파이프라인 API 라 파라메트릭 조립에
부적합), **Blender bpy**(§9 에서 따로 설명). 메쉬를 "numpy 로 조립해서 검증하고 내보내는"
용도라면 trimesh 가 사실상 유일한 표준이다.

**어떻게** — 모든 파트는 `Assembly.add` 를 지나며 4단계 정리를 받는다: 퇴화면 제거 → 정점
병합 → 미사용 정점 제거 → 법선 바깥 정렬(`fix_normals` + 부피 음수면 `invert`)
← 출처: src/cadkit.py:156 ("여기서 걸러야 PO/SBR 의 조명판정(n̂·û>0)이 오염되지 않는다").
각진 로프트는 `smooth`(Taubin 스무딩 + 선택적 서브디비전)로 실물처럼 다듬는다
← 출처: src/cadkit.py:446. 그 매끈함에는 **대가가 있다** — §6.1 에서 잰다.

## 5. manifold3d — 견고한 불리언 엔진

**무엇** — 3D 불리언(합집합·차집합·교집합) 전용 C++ 엔진의 파이썬 바인딩(버전
3.5.2 ← 출처: 실제 설치 환경; Apache-2.0, github.com/elalish/manifold).
trimesh 가 `engine="manifold"` 로 백엔드로 호출한다 ← 출처: src/cadkit.py, 73.

**왜 불리언이 어려운가** — 종이공작에 비유하면, 두 입체의 합집합은 "서로 뚫고 들어간 종이
상자 두 개를 교선(交線)을 따라 정확히 오려 붙이는" 작업이다. 교선 계산은 부동소수점 오차에
극도로 민감해서, 순진한 구현은 (a) 미세한 틈이 남아 watertight 가 깨지거나 (b) 면이 겹쳐
비다양체(non-manifold: 한 모서리에 면이 3개 이상 붙는 등 물리적으로 불가능한 상태)가 되기
일쑤다. 불리언은 **메쉬 연산 중 가장 잘 깨지는 연산**이다.

**왜 manifold 인가** — 이 엔진은 이름 그대로 "입력이 다양체(manifold)면 출력도 항상
다양체"를 설계 목표로 하는 최신 엔진이고(견고성 보장), 병렬화로 빠르며, trimesh 가 공식
백엔드로 지원해 `pip install manifold3d` 한 줄로 붙는다. 검토했던 대안: **Blender 엔진**
(trimesh 의 다른 백엔드; 외부 프로그램 설치·프로세스 호출 필요), **OpenSCAD/CGAL**(정확하지만
매우 느리고 의존성이 무겁다). 서버에 GUI 없이 pip 만으로 재현 가능해야 하므로 manifold 가
유일한 실용해였다.

**어떻게** — 두 군데서 쓴다 ← 출처: src/cadkit.py:184 이하:

* `Assembly.union_group` — 한 그룹 안의 파트들을 합집합으로 녹여 **겹친 부분의 내부 면을
  제거**한다 ("PO/SBR 이 헛세지 않는다").
* `Assembly.subtract` — 그룹에서 공구(tool) 메쉬를 **깎아낸다**(리세스·홈·구멍).

⚠ **지금 상태로 정확히 적을 것 두 가지.**

1. **합집합이 실패해도 예외도 로그도 안 남는다.** `union_group` 이 예외를 통째로 삼킨다
   — «안전한 후퇴» 가 아니라 **조용한 실패**다. 합집합이 안 된 그룹은 내부 면이 그대로
   남고, PO 는 가림을 안 보므로 그 면적을 이중으로 센다.
2. **manifold 로 끝나지 않는다.** 합집합 **결과 자체는 수밀**인데, 그 뒤 우리 코드가
   퇴화면(아주 가느다란 삼각형)을 지우는 단계에서 구멍이 날 수 있다. 삼각형을 그냥 지우면
   그 자리에 테두리(경계 모서리)가 남기 때문이다.
   그래서 정본에서는 지우는 대신 **모서리 붕괴**(collapse)로 없앤다 — 짧은 변의 두 끝점을
   하나로 합쳐 삼각형을 «접어» 버리는 방식이라 구멍이 안 생긴다
   ← 출처: `src/geom.py` 수리 `i5`(정본, §0.2) · `outputs/mesh_layer2_holes_poles_0816.json`.

   지금 상태: mini2 셸(body 그룹)은 면 7,756장 ·
   부품 1개 ·
   수밀 1/1 ·
   경계 모서리 **0개**다
   ← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` `A_geometry.mini2.groups.body`.
   함대 전체로도 경계 모서리는 **0개** —
   즉 구멍이 하나도 없다.

   ⭐ **그래도 «합집합만 하면 안전하다» 로 읽지 말 것.** 붕괴는 정점을 0.2 mm 옮긴다.
   그 정도 이동이 허용되는 이유는 그 자리가 «이웃이 3개뿐인 불리언 부산물» 이라는 것을
   따로 확인했기 때문이지, 일반적으로 안전해서가 아니다.

### 5.1 그룹 **사이**의 겹침 — 자를 세 번 갈았다

동체 속에 박힌 배터리·기판처럼 그룹 사이 겹침은 **의도된 설계**다. 문제는 «얼마나 겹치나» 를
어떤 자로 재느냐다. 자에 따라 우선순위가 뒤집힌다:

| 자 | 무엇을 세나 | 왜 부족한가 |
|---|---|---|
| ① 부피 % | 교차 부피 ÷ 전체 부피 | 산란은 부피가 아니라 **표면**에서 난다 |
| ② 표면적 % | 다른 부품 속에 묻힌 면적 비율 | 금속 1 cm² 와 플라스틱 1 cm² 를 같게 센다 |
| ③ **재질 가중 + 담는 쪽의 불투명 여부** | A·\|Γ\|² 로 세고, **유전체 셸 안**은 빼고 **불투명 부품 안**만 센다 | 지금 쓰는 자 |

③ 이 맞는 이유: 금속 상자가 **플라스틱 셸 안**에 있는 것은 결함이 아니라 설계다 —
전파는 셸을 투과해 그 금속을 본다. 진짜 이중계상은 **불투명한 부품 안**에 묻힌 면이다.

| 기체 | 총 매몰 [%] | 설계 의도 [%] | **진짜 결함** [%] | 예산 [%] | 재질 가중 불투명 안 [%] | PO 과대계상 [dB] |
|---|---|---|---|---|---|---|
| Mini 5 Pro | 38.8 | 12.4 | **26.3** | 36.2 | 9.59 | **+0.44** |
| Mavic 4 Pro | 34.6 | 12.5 | **22.2** | 32.7 | 8.92 | **+0.41** |
| Matrice 4E | 40.8 | 21.4 | **19.4** | 20.6 | 2.23 | **+0.10** |
| S1000+ | 8.3 | 1.1 | **7.2** | 7.6 | 0.69 | **+0.03** |
| Phantom 4 | 35.6 | 12.0 | **23.6** | 33.6 | 10.01 | **+0.46** |
| Typhoon H (H480) | 22.9 | 8.7 | **14.2** | 15.4 | 3.91 | **+0.17** |
| X500 V2 | 18.1 | 0.0 | **18.1** | 20.0 | 6.76 | **+0.30** |
| Phantom 3 Professional | 25.1 | 17.9 | **7.2** | 9.2 | 0.80 | **+0.04** |
| Matrice 350 RTK | 30.1 | 21.0 | **9.0** | 9.8 | 2.54 | **+0.11** |
| Mini 2 | 39.2 | 13.2 | **26.0** | 37.7 | 20.14 | **+0.98** |

← 출처: 면적 자(앞 네 열)는 `report_mesh/outputs/mesh_verify_canon_0817.json` `buried_faces`(검사기 `mesh_check.check_buried_faces`), 재질 가중 자(뒤 두 열)는
`report_mesh/outputs/mesh_canon_0817.json` `material_weighted`(fc = 3.5 GHz).

함대에서 가장 큰 것이 Mini 2(+0.98 dB), 가장 작은 것이
S1000+(+0.03 dB)다.

⭐ **이 표가 주는 교훈**: «묻힌 면적이 40 %» 라는 문장과 «PO 가 0.5 dB 과대» 라는 문장은
같은 사실의 두 얼굴이고, **결정을 내릴 때 쓸 것은 뒤쪽**이다. 앞쪽만 보면 우선순위를 잘못 잡는다.

⚠ **이 검사는 자기가 이름 붙인 결함에 대해 실패할 수 없다.** 잣대가 «전체 표면적 대비 비율»
이라, 부품 하나를 통째로 셸 속에 밀어 넣어도 예산을 못 넘고 값이 거꾸로 가기도 한다
(mini5pro 실측: 착륙다리를 통째로 묻으면 +1.59 pp, 모터를 묻으면 **−1.48 pp**).
지금 이 예산이 하는 일은 «오늘보다 나빠지지 않았다» 의 감시뿐이다
← 출처: `docs/MESH_CERTIFICATE.md` §1-③.

In [ ]:
# 불리언 합집합 데모 — 반쯤 겹친 상자 두 개를 하나의 껍질로 녹인다
import trimesh
a = trimesh.creation.box(extents=(1, 1, 1))          # 1x1x1 상자
b = trimesh.creation.box(extents=(1, 1, 1))
b.apply_translation([0.5, 0.5, 0.0])                 # 반쯤 겹치게 이동
u = trimesh.boolean.union([a, b], engine="manifold")  # manifold3d 가 실제 계산

print("합치기 전 면 수 :", len(a.faces) + len(b.faces))
print("합친 후  면 수 :", len(u.faces), "(교선을 따라 새 삼각형이 생김)")
print("watertight?    :", u.is_watertight)
print(f"부피           : {u.volume:.4f}  (겹침 0.5*0.5*1=0.25 가 한 번만 계산돼 1+1-0.25=1.75)")

## 6. scipy — 단면을 매끈하게 잇는 스플라인

**무엇** — 과학계산 라이브러리(버전 1.18.0 ← 출처: 실제 설치 환경; BSD, scipy.org).
이 프로젝트에서는 단 하나, `scipy.interpolate.CubicSpline`(3차 스플라인 보간)만 쓴다
← 출처: src/cadkit.py, 197-212 (`spline_sections`).

**왜** — 동체의 폭·높이를 제어점 6개로만 지정하고 그 사이를 **매끈한 곡선**으로 채우고 싶다.
직선(선형) 보간은 제어점마다 접선이 불연속이라 꺾인 자국이 남는다. CubicSpline
은 기울기까지 연속(C²)이라 꺾임이 없고, 자동차 보닛처럼 부드러운 실물 곡면이 나온다.
자작 스플라인은 수치 안정성 검증이 부담이라 검토조차 하지 않았다 — 이건 바퀴의 재발명이다.

**어떻게** — `spline_sections(xs, half_w, half_h, z_off)` 가 제어점을 CubicSpline 세 개(폭·높이
·중심 z)로 보간해 **초타원 단면 24~30장**을 뽑고, 이것이 그대로 `loft()` 의 입력이 된다
← 출처: src/cadkit.py:325. `z_off` 덕에 "코가 살짝 처지는" 실물 특징(nose drop)도 낸다
← 출처: src/drone_cad.py:634 (`_body_folding` 의 `zo` 배열).

### 6.1 매끈함에는 대가가 있다 — 두 가지, 둘 다 «밝게» 쪽으로 틀린다

«매끈해서 좋다» 만 적으면 절반이다. 지금 상태에서 실제로 재 본 두 가지를 적는다.

**① 스플라인이 제어점 사이에서 넘친다(overshoot).** 3차 스플라인은 C² 연속을 지키느라
제어점 사이에서 값을 살짝 벗어난다. 형상표가 «잘록한 허리 → 넓은 어깨» 로 꺾이면 특히 그렇다.
실측:

| 기체 | 무엇 | 형상표 의도 | 메쉬 실측 | 어긋남 |
|---|---|---|---|---|
| matrice4e | 셸 배(아래쪽) | −30.38 mm (형상표) · −30.81 mm (공식 CAD) | −34.74 mm | **CAD 대비 3.93 mm 더 아래** |
| mini5pro | 셸 최대 반폭 | 35.21 mm | 37.87 mm | **+7.6 %** |
| mini2 | 셸 높이 | 44.80 mm | 48.36 mm | **+8.0 %** |

⭐ **부호가 늘 같은 쪽이다** — 항상 «더 크게(= 전파에 더 밝게)». 우연이 아니라 스플라인
넘침의 성질이다. 고치는 길은 둘이다: 제어점을 6 → 8~10 으로 늘리거나, 넘치지 않는 보간
(PCHIP, 단조 스플라인)으로 바꾸는 것. 둘 다 전 기종 메쉬가 바뀌므로 별도 라운드의 일이다.

**② Taubin 스무딩이 끝단 캡을 안쪽으로 당긴다.** 로프트의 앞뒤 마감면은 삼각형 팬으로
닫혀 있는데, 스무딩이 그 테두리 링을 안으로 끌어당긴다. 실측(스무딩 0회 ↔ 4회 대조):

| 기체 | 어디 | 스무딩 전 | 스무딩 후 | 남은 비율 |
|---|---|---|---|---|
| matrice4e | 기수 단면(반폭×반높이) | 41.02×36.88 mm | 23.42×21.31 mm | 약 57 % |
| mavic4pro | 꼬리 단면 | 42.63×28.57 mm | 24.58×16.43 mm | 약 57 % |
| phantom3 | 기수 단면 | 26.04×22.01 mm | 16.30×12.96 mm | 약 61 % |

**가운데 4개 스테이션은 형상표를 0.5 % 안에서 재현한다** — 손실은 끝단 전용이다.
방위평균 투영면적으로는 −0.00~−0.06 dB 밖에 안 움직이므로 **레벨 결함이 아니다.**
흔들리는 것은 «기수를 정면으로 봤을 때의 정반사 형상» 이고, 그 크기는 아직 커널로 안 쟀다
(평판극한 상한 10 dB 는 상한일 뿐이다 — §4.4 의 «모른다» 목록).

← 출처: `outputs/mesh_inspect_body_arms_0816.json` `findings`(로프트 끝단 캡·스플라인 넘침).
고칠 자리는 이미 선택 인자로 뚫려 있다: `_body_folding(..., smooth_iters=)` 와
`_SHELL_SHAPE[key]['smooth_iters']`. **기본값이 옛 값이라 지금 메쉬는 비트동일하다.**

## 7. `src/cadkit.py` 함수 도감 — 무엇을 만드는 함수인가

cadkit 은 위 라이브러리들을 조합해 **CAD 동사(verb)** 를 제공한다. 함수 하나 = 모델링 동작
하나다 ← 출처: 각 행의 소스 위치(모두 src/cadkit.py) 및 사용처(src/drone_cad.py):

| 함수 | 무엇을 만드나 | 어떻게 | 대가(있으면) | 드론에서의 사용처 |
|---|---|---|---|---|
| `superellipse` (:307) | 초타원 단면 | 지수 n 으로 타원↔둥근 사각 사이 조절 | — | 동체·캐노피 단면 (`_body_folding` drone_cad.py:634) |
| `rounded_rect` (:316) | 둥근 모서리 사각 단면 | shapely `buffer` 오프셋 | — | 팔(arm) 단면 (`_arm_folding` drone_cad.py:1301) |
| `spline_sections` (:325) | 매끈하게 보간된 단면열 | scipy CubicSpline ×3 | **제어점 사이 넘침** — 셸 배 3.93 mm·반폭 +7.6 %(§6.1) | 동체 제어점 6개→단면 24~30장 |
| `loft` (:276) | 단면열을 이은 곡면 껍질 | 단면 등간격 재샘플 후 옆면 결합+앞뒤 캡 | — | 동체·캐노피·프로펠러 날 |
| `sweep` (:346) | 경로를 따라 민 관 | 접선 + 최소회전 프레임 | — | 접이식 팔·착륙다리 |
| `revolve` (:386) | 회전체 | (r,z) 프로파일을 z 축 둘레로 회전, r=0 은 꼭짓점으로 접음 | — | 모터 벨·프롭 허브·RTK 돔 (`_motor_bell` drone_cad.py:132) |
| `smooth` (:446) | 매끈해진 메쉬 | Taubin 스무딩(+서브디비전) | **끝단 캡 단면의 약 43 % 손실**(§6.1) | 로프트 직후 동체/캐노피 |
| `box`/`cyl`/`sphere`/`capsule` (:457~) | 기본 입체 | trimesh.creation 래퍼 | — | 배터리·기판·짐벌 볼·다리 |
| `Assembly` (:150) | 그룹(재질)별 파트 바구니 | add(정리)→union_group/subtract(불리언)→check(검증)→to_geom(변환) | union 실패가 조용하다(§5) | 모든 기체의 최종 조립 |

(행 번호는 `inspect` 로 소스에서 직접 읽어 넣는다.)

핵심 규약 세 가지 ← 출처: src/cadkit.py 머리말:
모든 파트는 **watertight** 로 만들고(검증 가능해짐), 파트마다 **그룹 이름**을 달아 재질의 단일
진리원(materials.py)과 연결하고, 마지막에 **geom.Mesh 로 변환**해 기존 파이프라인에 넘긴다.

### 7.1 조립 순서로 보는 도구들 — build_stages

![build_stages](outputs/figures/build_stages.png)

**그림 1 — Mavic 4 Pro 파라메트릭 CAD 의 조립 순서** (파트 1개 = OBJ 1개 = Sionna 재질 1개)
← 출처: report_mesh/src/viz_mesh_reports.py `fig_build_stages()` 가 생성. 각 단계가 곧 위 도감의
함수들이다:

| 그림 속 단계 | 만든 함수 (도구) | 소스 |
|---|---|---|
| 1. body (superellipse loft) | `spline_sections`+`superellipse` → `loft` → `smooth` | drone_cad.py:634 `_body_folding` |
| 2. + canopy/battery lid | 같은 로프트 조합(더 납작한 돔) | drone_cad.py:1204 `_canopy` |
| 3. + motors | `revolve` — 아웃러너 모터 벨 회전체 | drone_cad.py:132 `_motor_bell` |
| 4. + internal battery & PCB | `box` (trimesh.creation.box 래퍼) — 반투명으로 내장 표시 | drone_cad.py `INTERNALS` |
| 5. + gimbal camera | `box`+`cyl` — 3렌즈 짐벌 블록 | drone_cad.py:1364 `_gimbal_hasselblad` |
| 6. + propellers | 익형 단면을 비틀며 `loft` (평면형은 기체마다 다르다 — §0.2) | drone_cad.py:464 `_airfoil` · 494 `_blade` |

(접이식 기종의 팔은 `sweep` 으로 만들어 body 그룹에 합쳐지므로 단계 1 에 이미 포함돼 있다.
Mavic 4 Pro 의 최종 그룹은 8개: battery, body, camera, canopy, gear, motor, pcb, prop
← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` `A_geometry.mavic4pro.groups`.)

이렇게 조립된 Mavic 4 Pro 는 정점 15,682개 · 삼각형 31,280개다
— 그중 프로펠러가 13,984장(44.7 %)이다
← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` `A_geometry.mavic4pro`·`prop_triangles`.

## 8. 검사기는 무엇을 보고, 무엇을 못 보나

라이브러리 선택의 가치는 **검사가 결함을 실제로 잡을 때** 증명된다. 그래서 이 절은
«무엇을 잡는다» 만큼 «무엇을 아직 못 잡는다» 를 같은 무게로 적는다.

### 8.1 웰딩과 수리는 다른 일이다 — 하나는 해야 하고 하나는 하면 안 된다

둘 다 «메쉬를 손본다» 로 뭉뚱그려지지만, 검사기에게는 정반대 성격이다.

| | 무엇을 하나 | 검사기가 해도 되나 |
|---|---|---|
| **웰딩** (`process=True`) | 같은 자리에 겹친 **정점**을 하나로 합친다. 새 형상을 만들지 않는다 | ✅ **해야 한다.** 우리 `geom.Mesh` 는 프리미티브마다 정점을 따로 쌓으므로, 웰딩 없이는 멀쩡한 부품도 전부 «비수밀» 로 나온다 |
| **수리** (`repair=True`) | 없는 **삼각형**을 지어 구멍을 메운다 | ⛔ **하면 안 된다.** 찾으려던 결함이 측정 직전에 사라진다 |

← 출처: `src/mesh_check.py` 머리말이 정확히 이 구별을 규약으로 적는다.

### 8.2 지금의 11 검사

| # | 검사 | 무엇을 묻나 |
|---|---|---|
| 1 | **수밀(watertight)** | 부품이 닫힌 껍질인가 |
| 2 | **경계 모서리** | 삼각형 하나만 쓰는 모서리 = 구멍의 테두리. **원칙은 0**, 예산으로만 예외 |
| 3 | **winding** | 이웃한 면이 같은 방향으로 감겼는가 |
| 4 | **법선 방향** | 닫힌 부품의 부호있는 부피가 양수인가 |
| 5 | **부호부피(원본 인덱스)** | trimesh 를 **전혀 안 거치고** 출하 인덱스에서 손계산 |
| 6 | **퇴화면 — 절대 + 상대** | 면적 잣대에 더해 **최소 내각 <0.5°** 슬리버를 센다 |
| 7 | **그룹 안 겹침** | 같은 그룹의 두 부품이 서로 파묻혔는가(PO 면적 이중계상) |
| 8 | **치수 대조** | 프롭 지름·로터 대각·공표 외형을 `DroneSpec` 의 수와 대조 |
| 9 | **손대칭성** | 로터별 날 비틀림 방향이 회전방향과 맞는가 — **거울상 기체 탐지** |
| 10 | **프롭↔모터 벨 관통** | 원통 근사(빠름) + 솔리드 내부판정(판정 기준) |
| 11 | **⭐ 매몰면 전수** | **전 부품쌍**에서 다른 부품 솔리드 안에 든 면적. «설계 의도»(셸 안의 배터리·기판)와 «진짜 결함» 을 갈라 뒤쪽만 예산에 건다 — `check_buried_faces` · `BURIED_FACE_BUDGET_PCT` |

1~7 은 **부품(연결요소) 단위**로 돈다. 드론은 프리미티브의 합집합이라 전체 메쉬는 원래
수밀이 아니고, 의미 있는 검사는 부품별 검사이기 때문이다.

**«통과» 는 «0» 이 아니라 «선언된 예산 안» 이다:**

| 예산 | 지금 걸려 있는 값 | 무엇을 뜻하나 |
|---|---|---|
| `BOUNDARY_EDGE_BUDGET_FIXED` | 기본 **0** — 예외 없음 | 구멍은 원칙적으로 없어야 한다. 정본에서는 예외 칸이 비어 있다 |
| `SLIVER_BUDGET_BLADE_LAW` | 6기체에 별도 값 — 실측 241~644 / 예산 266~709 (나머지 기체는 기존 `SLIVER_BUDGET` 실측 260~389) | 아주 뾰족한 삼각형 개수. 면적 비중이 0.0001~0.03 % 라 σ 에는 무해하고, 감시하는 이유는 법선이 수치적으로 불안정한데 PO 조명 판정이 `n̂·û>0` 이기 때문이다 |
| `PROP_BELL_SOLID_AREA_PCT_BLADE_LAW` | typhoonh480 6.5 % · m350rtk 5.4 % | 프로펠러가 모터 벨 솔리드 **안**에 든 면적 비율 |
| `GROUP_OVERLAP_BUDGET_FIXED` | 기본 0.1 % (battery 실측 0.0 %) | 같은 그룹 안에서 부품이 파묻힌 비율 |
| `BURIED_FACE_BUDGET_PCT` | 기종별 7.6~37.7 % (실측 7.2~26.3 %) | 다른 부품 솔리드 **안**에 든 면적 중 «진짜 결함» 몫 |
| `DIM_TOL_PCT` | 프롭 지름 1 % · 외형 1 % · 대각 3 % (mini5pro 예외 12 %) | 공표 숫자와의 허용 오차 |
| `HANDEDNESS_MIN_ABS` | 0.05 | 날 비틀림 지표의 최소 크기. 이보다 작으면 «비틀리지 않았다» 는 뜻이라 부호를 믿을 수 없다 |

예산 표는 «이만큼이 옳다» 가 아니라 **«지금 이만큼이다» 라는 선언**이다.
그래서 새로 생기는 결함은 예산을 넘겨 **실패한다** — 숨기지 않으면서 회귀를 막는 방식이다.

⭐ **예산이 «법칙별» 로 갈렸다.** 옛 표(`SLIVER_BUDGET`·`PROP_BELL_SOLID_AREA_PCT`)는 전부 옛 날 법칙(`legacy`)에서 잰 스냅샷이다. 정본은 기체마다 다른 평면형으로 로프트를 다시 뜨므로 씨접합 슬리버 수와 뿌리 겹침이 달라진다 — 결함이 는 것이 아니라 **다른 형상**이다. 그래서 표를 덮어쓰지 않고 **법칙을 키에 넣어** 따로 선언한다(`SLIVER_BUDGET_BLADE_LAW` 6행 · `PROP_BELL_SOLID_AREA_PCT_BLADE_LAW` 2행).

⭐ **값은 실측 + 10 % 로만 두고 실측치를 괄호에 남긴다.** «예산을 올려 통과시킨다» 는 인증서가 이름 붙인 안티패턴이라, 얼마를 올렸는지 소스에서 바로 읽히게 한다 ← 출처: `src/mesh_check.py` 두 표의 주석 · `docs/MESH_CERTIFICATE.md` §1-③.

### 8.3 게이트가 어디에 걸려 있나 — 범위를 정확히

| 검사기 | 언제 도나 | 무엇을 보나 | 범위·단서 |
|---|---|---|---|
| `src/cadkit.py` `Assembly.check` | 빌드 도중 | 파트 하나를 붙일 때마다 | 부품 단위 수밀·법선 |
| `src/mesh_check.py` | 출하 게이트 | 11 검사 + 예산표 | `python src/drones.py`(OBJ 내보내기) 한 문에 배선. `MESH_GATE=off` 로 끌 수 있고, RCS·렌더가 쓰는 **인메모리 `build_drone()` 은 이 문을 안 지난다** |
| `report_mesh/src/verify_mesh_suite.py` | 원장 생성 | A~I 9절 | 이 시리즈의 숫자를 만든다. I 절(SBR)만 GPU |
| `report_mesh/src/verify_mesh_canon_0817.py` | 원장 생성(정본) | A·B·C·D·F·G | 같은 잣대(위 스위트의 `sec_*`)로 **정본 판**을 다시 잰다. 전부 CPU |
| `benchmark/check_gimbal_sensors_0816.py` | 특수 검사 | 짐벌·센서 게이트 A~D | 부착·삼킴·선언초과·재질 민감도 |
| `benchmark/mesh_internal_metal_check.py` | 특수 검사 | 내부 금속 포함 판정 | «금속 상자가 정말 셸 안인가» |
| `benchmark/mesh_certify.py` | ⭐골든 봉인 | 형상·치수·예산·바깥참값·문·인증서 여섯 축 | «오늘이 어제와 같은가» 를 지킨다. 봉인 대조 약 9 초 · `--full` 약 295 초. ⚠지키는 것은 «안 바뀜» 이지 «옳음» 이 아니다 |

출하 게이트는 `python src/drones.py`(부위별 OBJ 내보내기) **한 문**에 걸려 있다. 단서 셋:

1. 환경변수 `MESH_GATE=off` 로 끌 수 있다.
2. RCS·렌더·마이크로도플러가 쓰는 **인메모리 `build_drone()` 은 이 문을 안 지난다** —
   전 기종 검사가 수십 초 걸려 import 시점에 걸지 않는다고 코드가 스스로 적는다.
3. ⇒ «메쉬를 쓰는 모든 경로가 검사를 통과한다» 는 지금 상태보다 강한 말이다.

### 8.4 아직 못 보는 것

- **동일평면 겹침** — 두 부품 표면이 정확히 같은 자리에 있으면 관통 검사가 못 본다. 9기체에서 34쌍이 그 상태다(가장 큰 것은 s1000plus body↔battery 16,745.8 mm²).
- **기종별 재질 분기** — `drone_gamma_map(spec, fc)` 이 `spec` 을 안 쓴다. 지금은 재질이 기체와 무관해서 맞지만, 기종별 재질이 생기는 순간 조용히 틀린 답을 준다.
- **PO 경로의 가림** — `rcs_po.py` 가 자기 docstring 에서 자기차폐·다중반사를 무시한다고 선언한다. 부품 속에 묻힌 면이 그 경로에서는 이중계상된다(재질 가중으로 +0.03~+0.98 dB · 가장 작은 것 s1000plus · 가장 큰 것 mini2).
- ⭐**출하한 파일 자체** — 검사는 메모리 배열에서 돌고 파일은 그 뒤에 쓰인다. 되읽어 같은 검사를 먹이면 10기체 중 2기체가 실패한다(1 µm 격자 반올림) ← 출처: `docs/MESH_CERTIFICATE.md` §1-①.
- ⭐**부품이 있는가** — 그룹이 통째로 사라져도 검사 10계열과 바깥 참값이 전부 조용하다 ← 출처: `docs/MESH_CERTIFICATE.md` §3.2.

뒤의 둘은 **인증서 라운드가 새로 찾은 것**이라 조금 더 풀어 적는다.

1. **검사한 물건과 출하한 물건이 다르다.** 검사는 메모리 배열에서 돌고, 파일은 그 뒤에 쓰인다.
   `geom.Mesh.write_obj` 가 좌표를 `%.6f`(미터 단위 소수 6자리 = **1 µm 격자**)로 반올림하므로,
   그 반올림이 정점 몇 쌍을 정확히 같은 자리로 붙일 수 있다. 되읽어 같은 검사를 먹이면
   matrice4e·mini2 의 파일이 **실패한다**(면적 0 삼각형 6장 · 비다양체 모서리 6개).
   최대 좌표 이동은 0.50 µm — 3.5 GHz 파장의 6백만분의 일이라 산란에는 아무 영향이 없다.
   무너지는 것은 크기가 아니라 **주장**이다: «퇴화면 0 장» 이 출하물에 대해서는 참이 아니다.
   ⚠ 이 경로는 실제로 쓰인다 — `scene_build.drone_parts()` 가 Sionna/Mitsuba 에 먹일 OBJ 를
   같은 writer 로 쓴다.
2. **부품이 통째로 사라져도 아무도 못 본다.** mini5pro 에서 canopy(표면적 8.4 %)와
   gear(2.1 %)를 지워 봤더니 검사 10계열과 바깥 참값 7행이 전부 조용했다. 좌우 짝을
   **한쪽만** 지우면 대칭 검사가 울지만, 그룹 **전체**를 지우면 대칭이 유지돼 조용하다.
   메쉬 안에 «이 그룹이 이만큼 있어야 한다» 는 정보가 없기 때문이다.

← 출처: `docs/MESH_CERTIFICATE.md` §1-① · §3.2.

### 8.5 ⭐ 검사기를 만들 때의 규율 — 양성 대조부터 통과시켜라

음성 대조(«문제 없는 것에 0 이 나온다»)만으로는 **«0 을 내는 검사기»** 와
**«0 이 맞는 대상»** 을 구별할 수 없다. 그래서 이 저장소의 잣대는 **쓰기 전에 두 방향으로**
**검증한다** — 일부러 결함을 심은 표본(양성 대조)에서 반드시 걸리고, 멀쩡한 표본(음성 대조)
에서 0 이 나와야 그 잣대를 실제 메쉬에 댄다.

| 잣대 | 어떻게 검증했나 |
|---|---|
| 자기교차(삼각형-모서리 관통) | 양성 대조 4종(교차 삼각형·관통 삼각형·회전 상자쌍 18 hits·구쌍 84 hits)에서 전부 검출, 음성 대조 5종에서 0 |
| 부품 부양(안 닿는 파트) | **양방향** 거리 + 내부 판정 — 단방향 정점→면 거리는 큰 상자가 가는 봉을 감쌀 때 «떠 있다» 로 읽는다 |

← 출처: `docs/MESH_AUDIT_0816.md` §③·
`outputs/mesh_inspect_body_arms_0816.json` `retracted_by_this_round`
(잣대를 갈면서 값이 바뀐 항목의 이력은 `docs/RETRACTION_LOG.md` 가 맡는다).

⭐ **이 규율이 함대 규모로 확장돼 있다.** 인증서 라운드가 결함을 일부러 심어 검사가 무는지를
전수로 돌렸다:

| 무엇 | 지금 값 |
|---|---|
| 적대 대조(양성·음성) | **184/184** 통과, 6스위트 |
| 봉인기(`benchmark/mesh_certify.py`) 자체 적대 시험 | **21/21**(양성 17 · 음성 4) |
| 인증 매트릭스 | 450칸 중 **실패 0** |
| 양성 대조가 **아직 없는** 축 | 여섯 — A7(프롭↔벨 원통 근사) · V7·Z1·Z3(인증서 지문 봉인) · G1(분절·자세 재현) · W1(검사기 사각지대 자기신고) |

← 출처: `docs/MESH_CERTIFICATE.md` §0·§1-② · `outputs/mesh_cert_matrix_0816.json` `summary`.
⭐ 마지막 줄이 규율의 핵심이다 — **«없다» 를 «없다» 라고 적는 것**. 양성 대조가 없는 축의
«통과» 는 «결함을 심어도 안 울 수 있다» 는 뜻이라 다른 축의 통과와 무게가 다르다.

**지금 상태의 사실들** — 위 규율을 지킨 잣대로 잰 값이다:

- 드론 10기 전부 검사 통과. 중복 정점 104개 · 미사용 정점 0개 ·
  경계 모서리 0개
  ← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` `A_geometry.*`.
- **자기교차 0 건** · **안쪽을 향하는 법선 0 개**(전 기체·전 그룹, trimesh 수리를 전혀 거치지
  않고 출하 인덱스에서 손계산한 부호부피가 모두 양수) · **프롭 스윕 지름 오차 −0.000 %**
  ← 출처: `docs/MESH_AUDIT_0816.md` §③.

In [ ]:
# 지금 어느 판이고, 그 판에 어떤 예산이 걸리는지 직접 읽어 본다
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
import geom, mesh_check as MC

print("메쉬 수리(정본) :", sorted(geom.mesh_fix_set()))
print("날 법칙(정본)   :", geom.blade_law_canon())
print()
# ⭐ 예산 표는 «법칙별» 로 갈려 있다 — 지금 걸리는 쪽을 본다
print("경계 모서리 예산 :", MC.BOUNDARY_EDGE_BUDGET_FIXED
      if geom.mesh_fix_enabled("i5") else MC.BOUNDARY_EDGE_BUDGET)
print("슬리버 예산(법칙):", MC.SLIVER_BUDGET_BLADE_LAW)
print("프롭↔벨(법칙)   :", MC.PROP_BELL_SOLID_AREA_PCT_BLADE_LAW)
print("매몰면 예산      :", MC.BURIED_FACE_BUDGET_PCT)
print("치수 허용오차   :", MC.DIM_TOL_PCT, "| 대각 예외", MC.DIM_DIAGONAL_TOL_PCT)

## 8-2. 형상을 안 바꾸면서 고칠 자리를 뚫는 법 — 선택 인자와 지문 검증

메쉬를 고치는 일에는 함정이 있다. **형상을 한 줄만 바꿔도 하류의 모든 σ·마이크로도플러**
**결과가 낡는다.** 그래서 «지금 고칠 준비» 와 «지금 고치기» 를 분리하는 규약을 쓴다:

1. 고칠 자리를 **선택 인자**로 뚫는다.
2. **처음에는 기본값을 옛 동작**으로 둔다(근거가 설 때까지).
3. 편집 전후 소스를 각각 불러 메쉬를 짓고 **지문**을 비교한다 —
   sha256(float32 정점 + int32 삼각형). 전 기종 비트동일이면 통과.

지금 뚫려 있는 자리들:

| 자리 | 무엇을 고칠 수 있게 되나 | 기본값 |
|---|---|---|
| `_body_folding(..., smooth_iters=)` | 끝단 캡 손실(§6.1)을 기종별로 | 옛 값 |
| `_gear_arm_spikes(..., inboard=)` (로터별 시퀀스 허용) | 다리 안쪽 치우침을 로터별로 | 옛 스칼라 |
| `geom.uv_sphere(..., weld_poles=)` | 극점 중복 정점 | 꺼짐 |
| `materials.make_material(..., strict=)` | 모르는 재질 키에 예외를 낼지 | 꺼짐(값 무변경) |

← 출처: `outputs/mesh_inspect_body_arms_0816.json` `code_changes`·
`outputs/mesh_inspect_materials_check_0816.json` `code_changes`.

### ⭐ 그리고 «착지» 라는 넷째 단계가 있다

위 3단계는 **고칠 준비**까지다. 준비만 하고 끝나면 리포트는 영원히 옛 형상을 설명한다.
그래서 근거가 선 것은 **기본값을 옮긴다** — 지금 `battery, i5` 와 날 법칙
`per_airframe` 가 그 상태다(§0.2). 방향이 뒤집혔다는 뜻이다:

| | 준비 단계 | 착지 뒤(지금) |
|---|---|---|
| 아무 인자도 안 주면 | 옛 동작 | **정본** 동작 |
| 스위치를 주면 | 새 동작을 켠다 | **옛 동작으로 되돌린다**(`MESH_FIX=none BLADE_LAW=legacy`) |
| 비트동일 시험이 지키는 것 | «아직 안 바뀌었다» | «옛 판을 그대로 되살릴 수 있다» |

⭐ **착지에는 이름표가 따라와야 한다.** 산출물 파일 이름에 `_mfixbatteryi5_blperairframe` 를 붙인다 —
안 붙이면 계산기가 옛 판 결과를 그대로 재사용하고, 재계산이 «건너뜀» 으로 끝난다.
두 판이 이름만으로 갈리게 하는 것이 규약이다
← 출처: `benchmark/elevation_sweep_md.py` 꼬리표 블록.

⭐ **예산도 같이 갈라진다.** 예산 표는 «그 형상에서 잰 스냅샷» 이라, 형상이 바뀌면 표를
덮어쓰는 대신 **법칙을 키에 넣어 따로 선언한다**(§8.2). 덮어쓰면 두 판을 같은 자로 못 잰다.

### 도구가 «모른다» 를 말하게 만들기

재질을 얻는 경로가 둘인데, 한쪽만 모르는 키를 막고 있었다. 지금은 양쪽 다 말한다:

- `gamma_po('nylon')` → **예외**(KeyError). 처음부터 막혀 있었다.
- `make_material('nylon', …)` → **경고**(RuntimeWarning) + `materials.UNKNOWN_KEY_FALLBACKS`
  에 기록. `strict=True` 를 주면 예외. **아는 키의 숫자는 하나도 안 바뀐다.**

⭐ 조용한 폴백은 «없는 값» 을 «그럴듯한 값» 으로 바꿔 놓는다. 그것이 이 저장소가 가장
경계하는 실패 방식이다 — **빈칸이 가짜 값보다 낫다.**

In [ ]:
# 정본 원장에서 전 기종의 기하 검증 요약 읽기 — 본문 수치의 원천
import json, os
P = os.path.join("outputs", "mesh_verify_canon_0817.json")
V = json.load(open(P, encoding="utf-8"))
print("판(꼬리표) :", V["_meta"]["file_tag"], "|", V["_meta"]["mesh_fix"],
      V["_meta"]["blade_law"])
print(f"{'드론':12s} {'정점':>8s} {'삼각형':>8s} {'그룹':>4s}  검증")
for k in V["_meta"]["drones"]:
    g = V["A_geometry"][k]
    ok = "PASS" if g["ok"] else "FAIL"
    print(f"{k:12s} {g['n_verts']:8,d} {g['n_faces']:8,d} {g['n_groups']:4d}  {ok}"
          f"  (중복정점 {g['dup_vertices']}, 미사용 {g['unused_vertices']})")
tot_v = sum(V['A_geometry'][k]['n_verts'] for k in V['_meta']['drones'])
tot_f = sum(V['A_geometry'][k]['n_faces'] for k in V['_meta']['drones'])
print(f"{'합계':12s} {tot_v:8,d} {tot_f:8,d}")

## 9. 왜 Blender 가 아니라 코드인가 — 그림 도구가 아니라 "스펙 공장"이 필요해서

Blender 로도 이 드론들을 **만들 수는 있다**. 하지만 우리에게 필요한 건 그림 도구가 아니라
**숫자(공식 스펙)를 넣으면 모양이 나오는 공장**이다. 모델은 한 번 그리고 끝나는 그림이 아니라,
제원·재질·검증과 맞물려 계속 재생산되는 **파이프라인의 부품**이기 때문이다. 공장이어야 하는
이유는 셋이다:

1. **스펙 → 모양이 자동이다.** 목표 치수는 DJI 공식 제원(`src/drones.py` 의 `DroneSpec` —
   대각선·프로펠러 지름·공식 외형 `envelope_mm` 등 ← 출처: docs/SPECS.md 의 제원+URL)에서
   **변수로** 들어간다. 제원이 바뀌면 숫자 하나 고치고 재실행하면 전 기종이 다시 나온다.
   GUI 모델은 치수 변경이 곧 수작업 재모델링이다.
2. **부위 = 재질이 자동으로 붙는다.** 파트마다 그룹 이름이 코드로 붙어, Sionna RadioMaterial
   과 PO 반사계수가 **한 곳(materials.py)** 에서 일관되게 연결된다 ← 출처: src/cadkit.py.
   수십 개 파트에 GUI 로 이름을 손으로 달면 오타 하나가 곧 재질 누락 사고다.
3. **재현과 검증이 자동이다.** 생성기 한 줄로 누가 돌려도 같은 바이트의 메쉬가 나온다 —
   실제로 지문(sha256) A/B 로 10종 **비트동일**을 확인한다(§8-2). "동체 폭 계수
   0.46→0.50" 같은 변경은 텍스트 한 줄로 보인다(바이너리 .blend 는 그렇게 못 본다).
   검사기는 §8.3 에 적은 범위에서 돈다.

같은 "코드 CAD" 축의 대안(OpenSCAD, CadQuery/build123d, Blender bpy 스크립팅)도 있지만,
numpy 배열로 삼각형을 직접 다루며 pip 만으로 깔리는 trimesh 생태계가 위 세 요구와 기존
파이프라인(geom.Mesh → RCS/렌더)에 가장 잘 맞물렸다.

## 10. 정리와 재현

| 층 | 도구 | 역할 한 줄 |
|---|---|---|
| 수치 기반 | numpy 2.5.2 | 모든 좌표·변환의 공용 언어 |
| 컨테이너 | src/geom.py | Mesh 컨테이너(v/f/g·그룹별 OBJ 저장) + 챔버·범용 프리미티브 |
| 2D 단면 | shapely 2.1.2 | 초타원·둥근사각 단면과 재샘플 |
| 3D 조립 | trimesh 5.0.0 | 컨테이너·프리미티브·스무딩·**검증** |
| 불리언 | manifold3d 3.5.2 | 견고한 union/difference 엔진 |
| 보간 | scipy 1.18.0 | CubicSpline 로 매끈한 단면열 |

(버전 ← 출처: 실제 설치 환경 `/workspace/.venvs/py312` 을 importlib.metadata 로 조회)

이 도구 상자로 만든 결과물: 드론 10기, 정점 151,327개 · 삼각형 301,506개,
검사 결과 전 기종 통과(«선언된 예산 안» 이라는 뜻 — §8.2)
← 출처: `report_mesh/outputs/mesh_verify_canon_0817.json` `A_geometry`.
이 수는 **정본 판**(`MESH_FIX=battery,i5` · `BLADE_LAW=per_airframe`, 꼬리표 `_mfixbatteryi5_blperairframe`)의 것이다.

**이 편의 한 줄** — 도구는 능력을 주지만 **기본값이 규약을 정하지는 않는다.**
기본값은 **우리가** 정하고(§0.2), 정한 뒤에는 그 사실을 이름표로 못 박는다(§8-2).
검사기가 무엇을 보는지도 우리가 명시해야 하고, 못 보는 것은 적어 둬야 한다(§8.4).


⚠ **원장의 I 절(SBR)은 이월된 값이다** — GPU 없이 돌린 갱신이 GPU 증거를 지우지 않게
직전 원장에서 옮겨 왔고 `stale: true` 로 표시돼 있다. 다른 절과 세대가 다르므로
면 수·σ 를 나란히 인용하지 말 것.

**재현 명령**

```bash
PY=/workspace/.venvs/py312/bin/python
cd /workspace/sionna

# 0) 지금 어느 판인가 — 아무것도 안 주면 정본이다
PYTHONPATH=src $PY -c "import geom; print(geom.mesh_fix_set(), geom.blade_law_canon())"

# 1) 정본 원장 재생성 — 이것부터. 전부 CPU. 원장이 레지스트리와 다르면 일부러 멈춘다.
PYTHONPATH=src:benchmark $PY report_mesh/src/verify_mesh_canon_0817.py   # A·B·C·D·F·G
PYTHONPATH=src:benchmark $PY report_mesh/src/mesh_canon_0817.py         # 스위치·예산·매몰
PYTHONPATH=src:benchmark $PY report_mesh/src/verify_mesh_suite.py --skip-sbr   # H 절(옛 원장)
# 2) 그림 재생성
PYTHONPATH=src:benchmark $PY report_mesh/src/viz_mesh_reports.py
# 3) 이 노트북 재생성
PYTHONPATH=src:benchmark $PY report_mesh/src/make_mesh02.py
# 4) 회귀 봉인 — 형상이 그대로인가 (약 9 초 · 전체 약 295 초)
PYTHONPATH=src:benchmark $PY benchmark/mesh_certify.py
PYTHONPATH=src:benchmark $PY benchmark/mesh_certify.py --full --jobs 6
# 5) 옛 판을 되살릴 때 (비트동일) — 산출물 이름에 꼬리표가 안 붙는다
MESH_FIX=none BLADE_LAW=legacy PYTHONPATH=src:benchmark $PY <스크립트>
```

**다음 편** → mesh03: 모든 숫자와 모델의 출처 — 공식 제원·공식 CAD·실기체 스캔·타사 CAD 와
라이선스. 이전 편 ← mesh01: 시리즈 개요와 전체 지도.